# Exemplo de uso — `i3_shift_lake` no JupyterHub

Fluxo completo: **discover** (descobre e persiste o modelo do schema) → **extract** (lê paginado e grava parquet particionado) → **transform** (carrega qualquer tabela como `DataFrame` pandas).

Este notebook assume que o pacote `i3_shift_lake` está disponível (repositório clonado no ambiente do JupyterHub).

In [ ]:
import sys

# Se o notebook estiver em notebooks/ na raiz do repo, adiciona a raiz ao path
sys.path.append("..")

import pandas as pd

from i3_shift_lake import discover, load_model, extract_all, TableLoader

## Conexão com o Redshift

O pacote não abre conexão sozinho: ele espera receber uma função `query(sql) -> pandas.DataFrame`. Ajuste a célula abaixo para a forma de conexão já usada no seu JupyterHub (aqui um exemplo com `redshift_connector`, lendo credenciais de variáveis de ambiente — nunca deixe usuário/senha hardcoded no notebook).

In [ ]:
import os
import redshift_connector

conn = redshift_connector.connect(
    host=os.environ["REDSHIFT_HOST"],
    database=os.environ["REDSHIFT_DATABASE"],
    user=os.environ["REDSHIFT_USER"],
    password=os.environ["REDSHIFT_PASSWORD"],
    port=int(os.environ.get("REDSHIFT_PORT", 5439)),
)


def query(sql: str) -> pd.DataFrame:
    with conn.cursor() as cursor:
        cursor.execute(sql)
        return cursor.fetch_dataframe()

In [ ]:
from i3_shift_lake import ping

# valida a conexao/implementacao de query() antes de rodar o pipeline inteiro
ping(query)

## Parâmetros do job

`PAGE_SIZE` e `NUM_BUCKETS` controlam o trade-off tempo x memória x custo: páginas maiores reduzem o número de round-trips ao Redshift (menos overhead, menos tempo de leader node), enquanto mais buckets paralelizam melhor a leitura posterior em pandas/Spark. Ajuste conforme o volume de cada schema.

In [ ]:
SCHEMA = "meu_schema"
CONFIG_DIR = "./config"                 # onde ficam o modelo e as queries persistidas (JSON)
CONTROL_DIR = "./control"               # checkpoints da carga incremental (um por schema/tabela)
OUTPUT_DIR = "./dados_extraidos"        # onde ficam os parquets da extracao completa
SAMPLE_OUTPUT_DIR = "./dados_amostra"   # diretorio separado para nao misturar com a extracao completa
SAMPLE_CONTROL_DIR = "./control_amostra"  # control_dir proprio: amostra nao deve mexer no checkpoint real
PAGE_SIZE = 100_000
NUM_BUCKETS = 32


## 1. Discover — descobrir o modelo do schema

Busca tabelas, colunas e contagem de linhas via `information_schema`, e já persiste o resultado em `CONFIG_DIR/<schema>.json`. Rode isso uma vez; nas próximas sessões você pode pular direto para a célula "fluxo alternativo" abaixo, sem gastar leader node do Redshift de novo.

In [ ]:
model = discover(query, SCHEMA, config_dir=CONFIG_DIR)

pd.DataFrame(
    [{"table_name": t, "columns": len(cols), "row_count": model.row_counts[t]} for t, cols in model.tables.items()]
).sort_values("row_count", ascending=False)

### Fluxo alternativo — retomar de uma configuração já salva

Em uma sessão nova (ex.: kernel reiniciado, outro dia), se `CONFIG_DIR/<schema>.json` já existe, não é preciso repetir o `discover` contra o Redshift: basta carregar o modelo salvo. Descomente a linha abaixo para usar esse caminho em vez da célula anterior.

In [ ]:
# model = load_model(SCHEMA, config_dir=CONFIG_DIR)
# model.tables_with_rows()

## 2. Extract — ler paginado e gravar parquet particionado

Só tabelas com ao menos 1 linha são lidas. Cada tabela é ordenada e paginada pela melhor chave disponível (`date_modified`/`date_created` + `id`/`id_c`, com fallback para `id`/`id_c`), e a query usada fica salva em `CONFIG_DIR/<schema>/<tabela>.json` para auditoria/reprodução.

### Opção dev/teste — extrair apenas uma amostra

Antes de rodar a extração completa (que pode ser grande e demorada), vale rodar com `sample_size` para validar o pipeline ponta a ponta rapidamente: cada tabela é limitada às suas primeiras `sample_size` linhas, na mesma ordenação da extração completa.

**Importante**: como a escrita em parquet só acrescenta arquivos (nunca sobrescreve), use um `output_dir` separado (`SAMPLE_OUTPUT_DIR`) para a amostra. E como toda extração salva um checkpoint incremental, use também um `control_dir` próprio (`SAMPLE_CONTROL_DIR`) com `load_mode="full"` — assim a amostra nunca lê nem grava o checkpoint real da extração completa.


In [ ]:
relatorio_sample = extract_all(
    query, model, SAMPLE_OUTPUT_DIR, page_size=PAGE_SIZE, num_buckets=NUM_BUCKETS,
    config_dir=CONFIG_DIR, control_dir=SAMPLE_CONTROL_DIR, sample_size=1_000, load_mode="full",
)
relatorio_sample

In [ ]:
relatorio = extract_all(
    query, model, OUTPUT_DIR, page_size=PAGE_SIZE, num_buckets=NUM_BUCKETS,
    config_dir=CONFIG_DIR, control_dir=CONTROL_DIR,
)
relatorio

### Carga incremental x full

Por padrão (`load_mode="incremental"`), cada tabela guarda um checkpoint em `CONTROL_DIR/<schema>/<tabela>/checkpoint.json` com o último valor lido (data de controle + id). Rodar `extract_all` de novo lê só as linhas novas desde então — os parquets são sempre acrescentados, nunca sobrescritos, então o resultado final é o acumulado de todas as cargas.

Para reconstruir uma tabela do zero (ex.: mudou uma regra de negócio e o histórico precisa ser relido), use `load_mode="full"`: apaga os parquets e o checkpoint dessa tabela antes de reextrair.

In [ ]:
# rodar de novo: por default (incremental) so traz o que mudou desde a ultima vez
relatorio_incremental = extract_all(
    query, model, OUTPUT_DIR, page_size=PAGE_SIZE, num_buckets=NUM_BUCKETS,
    config_dir=CONFIG_DIR, control_dir=CONTROL_DIR,
)
relatorio_incremental

In [ ]:
# forcar reconstrucao completa de uma tabela (apaga parquets + checkpoint antes de reextrair)
# relatorio_full = extract_all(
#     query, model, OUTPUT_DIR, page_size=PAGE_SIZE, num_buckets=NUM_BUCKETS,
#     config_dir=CONFIG_DIR, control_dir=CONTROL_DIR, load_mode="full",
# )

## 3. Transform — trabalhar as tabelas extraídas em pandas

`TableLoader` encapsula a leitura dos parquets particionados; a partir daí é pandas puro.

In [ ]:
loader = TableLoader(OUTPUT_DIR, schema=SCHEMA, config_dir=CONFIG_DIR)
loader.available_tables()

In [ ]:
df_accounts = loader["accounts"]  # atalho equivalente a loader.load("accounts")
df_accounts.head()

In [ ]:
# Carregando só as colunas necessárias (menos I/O, mais rápido em tabelas largas)
df_leads = loader.load("custom_leads_c", columns=["id_c", "score"])
df_leads.describe()

## 4. Validate — checar a carga

Compara linhas carregadas com a contagem do `discover`, verifica duplicidade de `id`/`id_c` nos parquets, e dá espaço para checks de qualidade específicos do seu schema (chaves estrangeiras, chaves naturais únicas como cpf/cnpj).

In [ ]:
from i3_shift_lake import check_row_counts, check_duplicates_all

check_row_counts(model, OUTPUT_DIR)

In [ ]:
check_duplicates_all(model, OUTPUT_DIR, config_dir=CONFIG_DIR)

### Checks customizados (fks, chaves naturais como cpf/cnpj)

`check_unique`/`check_foreign_key` cobrem os casos comuns; para regras específicas, escreva sua própria função (mesma assinatura, retornando um `CheckResult`) e rode tudo junto com `run_checks`.

In [ ]:
from functools import partial
from i3_shift_lake import check_unique, check_foreign_key, run_checks

df_clientes = loader.load("clientes")
df_pedidos = loader.load("pedidos")

checks = [
    partial(check_unique, df_clientes, "cpf", "clientes"),
    partial(check_foreign_key, df_pedidos, "cliente_id", df_clientes, "id", "pedidos"),
]
run_checks(checks)

## Notas de custo/performance

- **Reexecução barata**: se `CONFIG_DIR` já tem o modelo e as queries, use `load_model` para pular o `discover` — evita bater no Redshift só para remontar o que já é conhecido.
- **Carga incremental é o default**: rodar `extract_all` periodicamente (ex.: 1x por dia) só traz o que mudou desde a última vez, em vez de reler a base inteira — a forma mais barata de manter os dados atualizados.
- **`PAGE_SIZE`**: valores maiores reduzem o número de queries (menos overhead de leader node), mas aumentam o pico de memória local por página lida.
- **`NUM_BUCKETS`**: mais buckets favorecem paralelismo na leitura posterior (Spark, Athena, etc.); para leitura só em pandas, 32 (default) costuma ser suficiente.
- **`loader.load(table, columns=[...])`**: carregue apenas as colunas necessárias em tabelas largas — o parquet é colunar, então isso reduz I/O de verdade, não só memória.
- **Um `control_dir`/`output_dir` por pipeline**: o checkpoint é identificado só por schema+tabela — reusar o mesmo `control_dir` para dois fluxos independentes da mesma tabela (ex.: uma amostra de teste e a carga real) faz um pisar no checkpoint do outro.
